# 🤟 Arabic Sign Alphabet Classifier — Deployment (Streamlit + Ngrok)
Serve your **trained MobileNetV2** Arabic sign model with a web UI on Colab.

### Files expected in Drive folder (edit path in Cell 1):
- `arabic_sign.keras`  ← primary (full saved model)
- `arabic_sign.h5`     ← fallback (optional)
- `arabic_sign.pkl`    ← metadata (`label2idx`, `idx2label`, `class_names`, `img_size`)

### Model metadata (from pkl):
- **Architecture:** MobileNetV2 backbone + custom head
- **Classes:** 28 Arabic sign alphabet letters
- **Input:** 224×224×3 RGB (preprocess: MobileNetV2 `preprocess_input`)

Run cells top → bottom. Update Drive path (Cell 1) and Ngrok token (Cell 5) before launching.

In [1]:
# =============================================================
# Cell 1 — Mount Drive & Copy Files
# =============================================================
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# ✏️ Change this to your Google Drive folder containing the artifacts
SRC_DIR = '/content/drive/MyDrive/arabic_sign/'

FILES = ['arabic_sign.keras', 'arabic_sign.h5', 'arabic_sign.pkl']

for fname in FILES:
    src_path = os.path.join(SRC_DIR, fname)
    if os.path.exists(src_path):
        shutil.copy(src_path, '.')
        print(f'✅ Copied: {fname} — {os.path.getsize(fname)/1024/1024:.2f} MB')
    else:
        print(f'⚠️  Not found (skipping): {fname}')

# Guards
if not os.path.exists('arabic_sign.keras') and not os.path.exists('arabic_sign.h5'):
    raise FileNotFoundError('❌ Neither arabic_sign.keras nor arabic_sign.h5 found — check your Drive path.')
if not os.path.exists('arabic_sign.pkl'):
    raise FileNotFoundError('❌ arabic_sign.pkl not found — needed for class names and metadata.')

print('\n✅ All required files ready!')

Mounted at /content/drive
✅ Copied: arabic_sign.keras — 36.78 MB
✅ Copied: arabic_sign.h5 — 36.42 MB
✅ Copied: arabic_sign.pkl — 18.52 MB

✅ All required files ready!


In [2]:
# =============================================================
# Cell 2 — Install Dependencies (use built-in TensorFlow)
# =============================================================
!pip install -q streamlit pyngrok pillow
import tensorflow as tf
print('Using built-in TF:', tf.__version__)
print('✅ Packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.5 MB/s eta 0:00:00
Using built-in TF: 2.20.0
✅ Packages installed!


In [3]:
# =============================================================
# Cell 3 — Verify Model Loads Correctly
# =============================================================
import pickle, numpy as np, os, warnings
warnings.filterwarnings('ignore')
from tensorflow import keras

# Load model (.keras preferred, .h5 fallback)
if os.path.exists('arabic_sign.keras'):
    model = keras.models.load_model('arabic_sign.keras', compile=False)
    print('✅ Loaded: arabic_sign.keras')
else:
    model = keras.models.load_model('arabic_sign.h5', compile=False)
    print('✅ Loaded: arabic_sign.h5')

# Load metadata
with open('arabic_sign.pkl', 'rb') as f:
    meta = pickle.load(f)

class_names = meta.get('class_names')
idx2label   = {int(k): v for k, v in meta.get('idx2label', {}).items()}
img_size    = meta.get('img_size', 224)
val_acc     = meta.get('val_accuracy', None)

# Sanity check
dummy = np.zeros((1, img_size, img_size, 3), dtype=np.float32)
pred  = model.predict(dummy, verbose=0)[0]
print(f"\n   Input size   : {img_size}×{img_size}×3")
print(f"   Classes      : {len(class_names)}")
if val_acc is not None:
    print(f"   Val accuracy : {val_acc*100:.2f}%")
print(f"   Test output  : first 3 probs → {pred[:3] if pred.ndim>1 else pred}")
print('\n✅ Model verified — proceed to Cell 4!')

✅ Loaded: arabic_sign.keras

   Input size   : 224×224×3
   Classes      : 31
   Val accuracy : 88.61%
   Test output  : first 3 probs → [3.2377608e-02 9.0372014e-05 1.1109124e-02 3.1450884e-03 8.3837350e-04
 9.0340397e-04 1.6673390e-02 2.7350806e-02 1.3794696e-02 4.6366458e-03
 5.2630017e-05 5.9024310e-03 9.4346981e-04 3.0012918e-03 6.4488972e-04
 2.5843931e-02 7.8656413e-03 2.5005888e-02 7.5985081e-02 1.8491569e-01
 1.7375411e-01 3.9706072e-03 1.7615901e-03 2.4806228e-02 9.6154697e-03
 8.2266884e-04 3.0158976e-02 2.2312686e-01 7.0131687e-04 1.2414930e-03
 8.8960148e-02]

✅ Model verified — proceed to Cell 4!


In [4]:
# =============================================================
# Cell 4 — Write the Streamlit App (no camera, no double preprocessing)
# =============================================================
app_code = r'''
import streamlit as st
import pickle, numpy as np, os, io, warnings, tensorflow as tf
from PIL import Image

warnings.filterwarnings("ignore")
st.set_page_config(page_title="Arabic Sign Classifier", page_icon="🤟", layout="wide")

@st.cache_resource
def load_all():
    # Load model (keras preferred, h5 fallback)
    if os.path.exists("arabic_sign.keras"):
        model = tf.keras.models.load_model("arabic_sign.keras", compile=False)
    else:
        model = tf.keras.models.load_model("arabic_sign.h5", compile=False)

    # Metadata and ordered labels
    with open("arabic_sign.pkl", "rb") as f:
        meta = pickle.load(f)
    idx2label = {int(k): v for k, v in meta.get("idx2label").items()}
    labels = [lbl for _, lbl in sorted(idx2label.items(), key=lambda kv: kv[0])]  # correct order
    img_size = meta.get("img_size", 224)
    val_acc = meta.get("val_accuracy")

    # Warm-up
    dummy = tf.zeros((1, img_size, img_size, 3), dtype=tf.float32)
    _ = model.predict(dummy, verbose=0)

    return model, labels, img_size, val_acc

def preprocess(pil_img, img_size):
    # Only scale to [0,1]; model already handles preprocess_input internally
    img = pil_img.convert("RGB").resize((img_size, img_size))
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, 0)

def predict(pil_img, model, labels, img_size):
    batch = preprocess(pil_img, img_size)
    probs = model.predict(batch, verbose=0)[0]
    idx = int(np.argmax(probs))
    return labels[idx], float(probs[idx]), probs

with st.spinner("Loading model (~20s first time, now warmed)..."):
    model, LABELS, IMG_SIZE, val_acc = load_all()

st.markdown("<h1 style='text-align:center'>🤟 Arabic Sign Classifier</h1>", unsafe_allow_html=True)
subtitle = "MobileNetV2 · 28 classes"
if val_acc is not None:
    subtitle += f" · Val Acc: {val_acc*100:.1f}%"
st.markdown(f"<p style='text-align:center;color:gray'>{subtitle}</p>", unsafe_allow_html=True)
st.markdown("---")

col1, col2 = st.columns([1,1], gap="large")

with col1:
    st.subheader("Upload a hand sign")
    uploaded = st.file_uploader("Choose an image", type=["jpg","jpeg","png","webp","bmp"], label_visibility="collapsed")
    use_img = Image.open(uploaded) if uploaded else None
    if use_img:
        st.image(use_img, caption="Selected image", use_column_width=True)
        st.caption(f"Resized to {IMG_SIZE}×{IMG_SIZE} for inference")

with col2:
    st.subheader("Prediction")
    if use_img:
        with st.spinner("Analysing..."):
            pred_cls, conf, probs = predict(use_img, model, LABELS, IMG_SIZE)
        st.success(f"Prediction: **{pred_cls}** — {conf*100:.2f}%")
        st.markdown("**Top 5 probabilities:**")
        top5 = sorted(zip(LABELS, probs), key=lambda x: x[1], reverse=True)[:5]
        for cls, p in top5:
            c1, c2, c3 = st.columns([2,5,1])
            c1.write(cls)
            c2.progress(float(p))
            c3.write(f"{p*100:.1f}%")
    else:
        st.info("Upload an image to get a prediction.")

st.sidebar.title("Model Info")
st.sidebar.write(f"Classes: {len(LABELS)}")
st.sidebar.write("Backbone: MobileNetV2")
if val_acc is not None:
    st.sidebar.write(f"Val Accuracy: {val_acc*100:.2f}%")
st.sidebar.caption("For demo/education. Not a clinical tool.")
'''

with open('arabic_sign_app.py', 'w') as f:
    f.write(app_code)

print('✅ arabic_sign_app.py written successfully!')

✅ arabic_sign_app.py written successfully!


In [5]:
# =============================================================
# Cell 5 — Launch App with Ngrok
# =============================================================
import subprocess, time
from pyngrok import ngrok, conf

# ✏️ Paste your Ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3Avo8t2NTG0xdg3ylUwsMHqzaI5_5LhJ6X28xPj9kRKgFYpg7'

if not NGROK_TOKEN or len(NGROK_TOKEN) < 10:
    raise ValueError('❌ Please set a valid Ngrok token.')

try:
    ngrok.kill()
except Exception:
    pass

conf.get_default().auth_token = NGROK_TOKEN

proc = subprocess.Popen(
    ['streamlit', 'run', 'arabic_sign_app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print('⏳ Waiting for Streamlit to start (~25–30s)...')
time.sleep(30)

public_url = ngrok.connect(8501)
print(f"\n{'='*60}")
print(f"🌐 Your app is live at: {public_url}")
print(f"{'='*60}")
print('\n⚠️  Keep this cell running. Stop it (or run Cell 6) to shut down the app.')

⏳ Waiting for Streamlit to start (~25–30s)...

🌐 Your app is live at: NgrokTunnel: "https://cribla-paleopsychic-freda.ngrok-free.dev" -> "http://localhost:8501"

⚠️  Keep this cell running. Stop it (or run Cell 6) to shut down the app.


In [ ]:
# =============================================================
# Cell 6 — Stop the App (run when done)
# =============================================================
from pyngrok import ngrok
ngrok.kill()
print('✅ Ngrok tunnel closed. You can stop the runtime now.')

# New Section